---
tags: [integration]
---

# HUGR Support

Compile a Qamomile Bell-state circuit to HUGR, an intermediate representation for quantum programs, and check it locally with Selene.

Install Qamomile with the `hugr` extra.

In [1]:
# Install Qamomile with HUGR support.
# !pip install "qamomile[hugr]"

In [2]:
import qamomile as qm
from qamomile.hugr import Hugr, Selene

## Build a circuit with Qamomile

Prepare two entangled qubits and measure them. The return value contains the two measurement bits.

In [3]:
@qm.qkernel
def bell() -> qm.Shot[qm.Bits[2]]:  # type: ignore[valid-type]
    state = qm.qubits(2)
    state = qm.h(state, target=0)
    state = qm.cx(state, control=0, target=1)
    return qm.measure(state)


program = qm.specialize(bell)
artifact = qm.compile(program, to=Hugr())

HUGR valid!


`qm.specialize(bell)` validates the program before choosing a target. `qm.compile(program, to=Hugr())` creates the compiled HUGR program used in the next step.

## Check the circuit locally

Run 100 shots with Selene. A shot is one repetition of the quantum program. In this noiseless simulation, the two measured bits always agree. The counts use tuples of `0` and `1` for the measurement results.

In [4]:
shots = 100
record = qm.run(artifact, on=Selene(), shots=shots, seed=7)
local_result = record.result
counts = dict(local_result.counts)

assert local_result.shots == shots
assert sum(counts.values()) == shots
assert set(counts) <= {(0, 0), (1, 1)}
print(counts)

{(0, 0): 51, (1, 1): 49}


## Summary

- Compile the circuit to HUGR with `qm.compile(program, to=Hugr())`.
- Check the circuit locally with `qm.run(artifact, on=Selene(), shots=100)` and read the measurement counts from `record.result.counts`.